# MyTennisCoach: tracking on a Colab GPU

The slow part of the pipeline (finding the ball and the players in every frame) runs here on a free GPU.
Everything else (hits, statistics, technique, video) runs at home in about a minute from the file this notebook produces.

**Before you start**

1. At home, drag the recording onto `prepare_colab.bat`. It fills `outputs\for_colab`.
2. Upload the contents of that folder to Google Drive, into `My Drive/MyTennisCoach/in`.
3. Here: menu **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**.

**Afterwards**

Download the `*_tracks.pkl` files from `My Drive/MyTennisCoach/out` into `F:\MyTennisCoach\outputs\cache`,
then drag the original recording onto `analyze.bat`.

In [ ]:
# 1. GPU check
import torch
assert torch.cuda.is_available(), "No GPU. Menu: Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
# 2. Google Drive (asks for permission once)
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
BASE = Path('/content/drive/MyDrive/MyTennisCoach')
IN, OUT = BASE / 'in', BASE / 'out'
IN.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
videos = sorted(IN.glob('*_work.*'))
print(len(videos), 'working copies found:')
for v in videos: print('  ', v.name, round(v.stat().st_size / 1e6), 'MB')

In [ ]:
# 3. Code and model weights (about 2 minutes)
%cd /content
!rm -rf MyTennisCoach
!git clone -q --recurse-submodules --shallow-submodules --depth 1 https://github.com/merexodram/MyTennisCoach.git
%cd /content/MyTennisCoach
# Colab already has PyTorch; install only what is missing.
!pip -q install ultralytics lap duckdb gdown "pandas<3"
!python scripts/download_weights.py

In [ ]:
# 4. Tracking. Progress is printed; an interrupted run resumes where it stopped if you run this cell again.
import os, shutil, subprocess, sys
os.environ['PYTHONPATH'] = '/content/MyTennisCoach/src'
CACHE = Path('/content/MyTennisCoach/outputs/cache'); CACHE.mkdir(parents=True, exist_ok=True)
for v in videos:
    name = v.stem[:-5]
    done = OUT / f'{name}_tracks.pkl'
    if done.exists():
        print('already done:', name); continue
    local = Path('/content') / v.name                     # reading from Drive directly is slow
    if not local.exists(): shutil.copyfile(v, local)
    court = IN / f'{name}.court.json'
    if court.exists(): shutil.copyfile(court, Path('/content') / court.name)
    print('=====', name)
    r = subprocess.run([sys.executable, '-u', '-m', 'mytenniscoach.process', str(local), '--track-only'])
    produced = CACHE / f'{name}_tracks.pkl'
    if r.returncode == 0 and produced.exists():
        shutil.copyfile(produced, done); print('saved to Drive:', done.name)
    else:
        print('FAILED:', name)

Done. The `*_tracks.pkl` files are in `My Drive/MyTennisCoach/out`.